# memory
> 

In [ ]:
#| default_exp memory

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from pyld import jsonld
import httpx
from rdflib import Graph, Dataset, URIRef, Namespace
from rdflib import Literal as RDFLiteral
from rdflib.namespace import RDF, RDFS
from datetime import datetime
from fastcore.basics import patch
from claudette import Chat, toolloop, models
import time
from pathlib import Path
from typing import List, Optional, Dict, Any
from dataclasses import dataclass, field
import pickle
from cogitarelink.vocabtools import register_vocab_aware_loader, compact_entity_with_vocabulary, VOCABULARY_REGISTRY

## Semantic Memory Module

- retriever_py (lowest level)

    ↑

- memory_py (imports retriever)

    ↑
    
- memtools_py (imports memory)


In [ ]:
from cogitarelink.retriever import *

## ToDo from previous memory.py.

### Components to Re-implement in 03_memory.ipynb

After analyzing both files, I've identified several important components that are present in the deprecated version but missing from the current 03_memory.ipynb. These need to be re-implemented:

#### 1. Named Graph Support
- `add_named_graph`: Add data as a named graph
- `query_named_graph`: Query within a specific named graph
- `list_named_graphs`: List all named graphs in the memory system

#### 2. Advanced Context Handling
- Enhanced `_register_contexts` method for better handling of scoped contexts
- `list_contexts`: List and inspect available contexts
- `search_with_contexts`: Search within specific contexts

#### 3. Enhanced URI and Linked Data Functions
- `follow_your_nose`: Linked data traversal function
- `follow_link`: Follow specific link relations
- `dereference_uri`: Advanced URI dereferencing with caching
- `resolve_did`: Support for DID (Decentralized Identifier) resolution

#### 4. Entity Creation and Management
- `create_entity_with_uuid`: Create new entities with UUIDs

#### 5. Enhanced Vocabulary Support
- `ensure_standard_vocabulary`: Ensure a standard vocabulary is loaded
- `load_vocabulary_content`: Parse vocabulary data into triples
- `detect_vocabulary`: Auto-detect which vocabulary an entity uses

#### 6. Container Support for JSON-LD 1.1
- `build_type_container`: Create JSON-LD type containers
- `build_property_container`: Build property-indexed containers

####7. LLM Integration Tools
- `select_relevant_contexts`: Select relevant contexts for LLM reasoning
- `format_context_for_llm`: Format context data for LLM consumption
- `generate_context_reasoning_hints`: Generate reasoning hints about contexts
- `answer_query_with_context_aware_tools`: Claude integration with context-aware tools
- `answer_query_with_graph_aware_tools`: Claude integration with graph-aware tools

#### 8. Advanced Graph Operations
- Graph partitioning improvements for complex context scenarios
- Enhanced relationship extraction between entities
- Support for handling multi-context documents like Verifiable Credentials

These components would significantly enhance the semantic memory capabilities, particularly for applications that need to handle multiple vocabularies, complex linked data traversal, and provide richer context to LLMs for reasoning tasks.

## Core semantic memory definition

In [ ]:
#| export
class SemanticMemory:
    def __init__(self, cache_dir=None, preload_vocabs=None):
        "Enhanced initialization with vocabulary support"
        self.dataset = Dataset()
        self.default_graph = self.dataset.graph(URIRef("urn:x-arq:DefaultGraph"))
        self.original_data = {}
        
        self.indices = {"by_id": {}, "by_type": {}, "by_label": {}, "by_keyword": {}, 
                    "by_graph": {}, "by_vocabulary": {}, "containers": {}}
        
        self.contexts = {}
        self.context_registry = {}
        
        self.client = httpx.Client(follow_redirects=True)
        
        self.cache_dir = cache_dir
        if cache_dir: Path(cache_dir).mkdir(exist_ok=True)
        
        register_vocab_aware_loader()
        
        self.preloaded_vocabs = set()
        if preload_vocabs:
            for vocab in preload_vocabs:
                self.preload_vocabulary(vocab)
        else:
            for vocab in ["schema", "dc", "foaf"]:
                self.preload_vocabulary(vocab)


## New unified way to add data to memory

In [ ]:
@patch 
def add(self:SemanticMemory, data, context=None, graph_id=None):
    "Enhanced add with vocabulary awareness"
    import copy, uuid
    
    if context and isinstance(data, dict):
        data = copy.deepcopy(data)
        if "@context" not in data: data["@context"] = context
        elif isinstance(data["@context"], list): data["@context"] = [context] + data["@context"]
        else: data["@context"] = [context, data["@context"]]
    
    vocabs = []
    if isinstance(data, dict) and "@context" in data:
        vocabs = detect_vocabularies_in_context(data["@context"])
        if vocabs: 
            self._track_vocabulary_usage(vocabs, data)
            
            for vocab in vocabs:
                if vocab not in self.preloaded_vocabs:
                    self.preload_vocabulary(vocab)
    
    if isinstance(data, dict) and "@graph" in data:
        if "@id" in data and not graph_id: graph_id = data["@id"]
        result = self._process_graph(data)
    elif self._needs_graph_partition(data):
        result = self._process_graph(self._create_graph_partition(data))
    else:
        result = self._process_entity(data)
    
    if graph_id and result:
        if isinstance(result, list):
            for entity in result: self._add_to_named_graph(entity, graph_id)
        else:
            self._add_to_named_graph(result, graph_id)
    
    return result


## Vocabulary Support

In [ ]:
#| export
@patch
def preload_vocabulary(self:SemanticMemory, vocab_name):
    "Preload a vocabulary to avoid HTTP requests during processing"
    try:
        context = load_context_for_vocabulary(vocab_name)
        self.contexts[vocab_name] = context
        self.preloaded_vocabs.add(vocab_name)
        return True
    except Exception as e:
        print(f"Error preloading vocabulary {vocab_name}: {e}")
        return False
    vocab_to_use = None
    if preferred_vocabs:
        for v in preferred_vocabs:
            if v in entity_vocabs:
                vocab_to_use = v
                break
    
    if not vocab_to_use: vocab_to_use = entity_vocabs[0]
    
    try:
        return compact_entity_with_vocabulary(entity, vocab_to_use)
    except:
        return entity

In [ ]:

#| export
@patch
def _track_vocabulary_usage(self:SemanticMemory, vocabs, data):
    "Track which vocabularies are used by this entity"
    entity_id = data.get("@id")
    if not entity_id: return
    
    # Ensure we have the vocabulary index
    if "by_vocabulary" not in self.indices:
        self.indices["by_vocabulary"] = {}
    
    # Track each vocabulary
    for vocab in vocabs:
        if vocab not in self.indices["by_vocabulary"]:
            self.indices["by_vocabulary"][vocab] = set()
        self.indices["by_vocabulary"][vocab].add(entity_id)

In [ ]:
#| export
@patch
def query_by_vocabulary(self:SemanticMemory, vocab_name, limit=50):
    "Find entities using a specific vocabulary"
    results = []
    
    if "by_vocabulary" in self.indices and vocab_name in self.indices["by_vocabulary"]:
        entity_ids = list(self.indices["by_vocabulary"][vocab_name])[:limit]
        for eid in entity_ids:
            entity = self.query_by_id(eid)
            if entity: results.append(entity)
    
    return results


In [ ]:
#| export
@patch
def compact_with_preferred_vocab(self:SemanticMemory, entity, preferred_vocabs=None):
    "Compact entity using preferred vocabulary"
    if not isinstance(entity, dict) or "@id" not in entity: return entity
    
    entity_vocabs = []
    if "by_vocabulary" in self.indices:
        for vocab, entities in self.indices["by_vocabulary"].items():
            if entity["@id"] in entities:
                entity_vocabs.append(vocab)
    
    if not entity_vocabs: return entity
    

### Old vocabulary support

In [ ]:
#| export
@patch
def compact_with_vocabulary(self:SemanticMemory, entity_id, vocab_name):
    """Retrieve an entity and compact it using a specific vocabulary"""
    entity = self.query_by_id(entity_id)
    if not entity: return None
    return compact_entity_with_vocabulary(entity, vocab_name)

In [ ]:
#| export
@patch
def use_vocabulary(self:SemanticMemory, vocab_name):
    """Load and register a vocabulary for use in this memory instance"""
    # This would call VocabTools functions but not depend on VocabTools internals
    context = load_context_for_vocabulary(vocab_name)
    return self._register_context(context)

## Named Graph Support

In [ ]:
#| export
@patch
def _add_to_named_graph(self:SemanticMemory, entity, graph_id):
    "Add processed entity to a named graph"
    import json
    from rdflib import Graph, URIRef
    
    # Skip if no entity ID
    if not isinstance(entity, dict) or "@id" not in entity: return
    
    # Get target graph
    graph_uri = URIRef(graph_id) if isinstance(graph_id, str) else graph_id
    target_graph = self.dataset.graph(graph_uri)
    
    # Add to RDF graph
    g = Graph()
    g.parse(data=json.dumps(entity), format="json-ld")
    target_graph += g
    
    # Update graph index
    if "by_graph" not in self.indices: self.indices["by_graph"] = {}
    if graph_uri not in self.indices["by_graph"]: self.indices["by_graph"][graph_uri] = set()
    self.indices["by_graph"][graph_uri].add(entity["@id"])


## Batch Processing

In [ ]:
#| export
@patch
def bulk_load(self:SemanticMemory, data_list, graph_id=None, detect_graph=True):
    "Bulk load multiple entities into memory"
    results = []
    
    # Process each item in the list
    for item in data_list:
        # Auto-detect graph ID if requested
        item_graph = None
        if detect_graph and isinstance(item, dict) and "@id" in item:
            item_graph = item.get("@graph", {}).get("@id", None)
        
        # Use provided graph ID if no auto-detected graph
        if not item_graph: item_graph = graph_id
        
        # Add to memory
        result = self.add(item, graph_id=item_graph)
        if result: results.append(result)
    
    return results


## Container Support

In [ ]:
#| export
@patch
def as_container(self:SemanticMemory, entities, container_type="@id", property_name=None, base_context=None):
    "Create a JSON-LD 1.1 container from entities"
    if not entities: return {}
    
    # Create container structure with proper JSON-LD 1.1 context
    result = {"@context": {"@version": 1.1}}
    if base_context: 
        if isinstance(base_context, dict): result["@context"].update(base_context)
        else: result["@context"]["@vocab"] = base_context
    
    # Set up the container definition
    if property_name:
        result["@context"][property_name] = {"@container": container_type}
        result[property_name] = {}
        container = result[property_name]
    else:
        container = {}
    
    # Process based on container type
    if container_type == "@id":
        for entity in entities:
            if "@id" not in entity: continue
            container[entity["@id"]] = entity
    elif container_type == "@type":
        for entity in entities:
            if "@type" not in entity: continue
            types = entity["@type"] if isinstance(entity["@type"], list) else [entity["@type"]]
            for t in types:
                if t not in container: container[t] = []
                container[t].append(entity)
    elif container_type == "@language":
        # Group content by language
        lang_map = {}
        for entity in entities:
            for prop, values in entity.items():
                if prop.startswith("@"): continue
                if isinstance(values, list):
                    for v in values:
                        if isinstance(v, dict) and "@value" in v and "@language" in v:
                            lang = v["@language"]
                            if lang not in lang_map: lang_map[lang] = []
                            lang_map[lang].append({"@id": entity["@id"], prop: v["@value"]})
                
        container.update(lang_map)
    
    # If we used a property name, make sure it's in the result
    if property_name and container:
        result[property_name] = container
    elif not property_name:
        # If no property name, add container directly to result
        for k, v in container.items(): result[k] = v
    
    return result

In [ ]:
#| export
@patch
def as_language_container(self:SemanticMemory, query, text_properties=None):
    "Create a language map from entities with multilingual text"
    entities = self.search(query) if isinstance(query, str) else query
    return self.as_container(entities, container_type="@language")

In [ ]:
#| export
@patch
def as_type_container(self:SemanticMemory, query=None, type_uri=None):
    "Create a type map grouping entities by type"
    if type_uri:
        entities = self.query_by_type(type_uri)
    else:
        entities = self.search(query) if isinstance(query, str) else query
    return self.as_container(entities, container_type="@type")



In [ ]:
#| export
@patch
def as_id_container(self:SemanticMemory, query=None, id_list=None):
    "Create an ID map for direct entity access"
    if id_list:
        entities = [self.query_by_id(id) for id in id_list if self.query_by_id(id)]
    else:
        entities = self.search(query) if isinstance(query, str) else query
    return self.as_container(entities, container_type="@id")


In [ ]:
#| export
@patch
def as_graph_container(self:SemanticMemory, graph_ids=None):
    "Create a @graph container grouping entities by graph"
    result = {"@context": {"@version": 1.1}, "@graph": {}}
    
    # Get all graphs if none specified
    if not graph_ids:
        graph_ids = [str(g.identifier) for g in self.dataset.graphs() 
                   if str(g.identifier) != "urn:x-arq:DefaultGraph"]
    
    # Process each graph
    for graph_id in graph_ids:
        entity_ids = set()
        if "by_graph" in self.indices and URIRef(graph_id) in self.indices["by_graph"]:
            entity_ids = self.indices["by_graph"][URIRef(graph_id)]
        
        # Get entities and add to container
        if entity_ids:
            graph_entities = []
            for eid in entity_ids:
                entity = self.query_by_id(eid)
                if entity: graph_entities.append(entity)
            
            if graph_entities:
                result["@graph"][graph_id] = graph_entities
    
    return result


In [ ]:
#| export
@patch
def store_container(self:SemanticMemory, container, container_id=None):
    "Store a container structure with container-aware indexing"
    if not container_id: 
        import uuid
        container_id = f"urn:container:{uuid.uuid4()}"
    
    # Ensure container has an ID
    if "@id" not in container: container["@id"] = container_id
    
    # Track as a container
    if "containers" not in self.indices: self.indices["containers"] = {}
    self.indices["containers"][container_id] = container
    
    # Add to memory
    return self.add(container, graph_id=container_id)


In [ ]:
#| export
@patch
def get_container(self:SemanticMemory, container_id):
    "Retrieve a previously stored container"
    if "containers" in self.indices and container_id in self.indices["containers"]:
        return self.indices["containers"][container_id]
    return None

## Below Here is the old code

## Core Processing Functions

In [ ]:
#| export
@patch
def _process_graph(self:SemanticMemory, data):
    """Process a graph structure with multiple entities"""
    if not isinstance(data, dict) or "@graph" not in data: return None
    
    results = []
    shared_ctx = data.get("@context")
    
    # Process each entity
    for entity in data["@graph"]:
        # Skip bare references
        if isinstance(entity, dict) and len(entity) == 1 and "@id" in entity: continue
        
        # Apply shared context if needed
        if shared_ctx and isinstance(entity, dict) and "@context" not in entity:
            entity = dict(entity, **{"@context": shared_ctx})
        
        # Process entity
        result = self._process_entity(entity)
        if result: results.append(result)
    
    return results

In [ ]:
#| export
@patch
def _process_entity(self:SemanticMemory, data):
    """Process a single entity, adding it to indices and graph"""
    import copy, json, uuid
    from rdflib import Graph
    
    if not isinstance(data, dict): return None
    
    # Normalize the entity
    entity = self._normalize_entity(copy.deepcopy(data))
    
    # Ensure ID
    if "@id" not in entity: entity["@id"] = f"urn:uuid:{uuid.uuid4()}"
    entity_id = entity["@id"]
    
    # Store original data
    self.original_data[entity_id] = entity
    
    # Register context and track associations
    if "@context" in entity:
        ctx_id = self._register_context(entity["@context"])
        if ctx_id:
            ctx_info = self.context_registry.setdefault(ctx_id, {"used_by": set()})
            ctx_info["used_by"].add(entity_id)
    
    # Try expanding and indexing
    try:
        expanded = jsonld.expand(entity)
        
        # Update indices
        self._update_indices(expanded)
        
        # Add to RDF graph
        g = Graph()
        g.parse(data=json.dumps(expanded), format="json-ld")
        self.default_graph += g
        
        return expanded
    except Exception as e:
        # Fallback to direct indexing
        print(f"JSON-LD expansion failed: {e}")
        self.indices["by_id"][entity_id] = entity
        
        if "@type" in entity:
            types = [entity["@type"]] if not isinstance(entity["@type"], list) else entity["@type"]
            for t in types:
                if t not in self.indices["by_type"]: self.indices["by_type"][t] = []
                self.indices["by_type"][t].append(entity)
        
        return entity


In [ ]:
#| export
@patch
def _needs_graph_partition(self:SemanticMemory, data):
    """Determine if data needs graph partitioning"""
    if not isinstance(data, dict): return False
    if "@context" in data and isinstance(data["@context"], list) and len(data["@context"]) > 1: return True
    if "credentialSubject" in data and isinstance(data["credentialSubject"], dict): return True
    return False

In [ ]:
#| export
@patch
def _create_graph_partition(self:SemanticMemory, data):
    """Convert document into graph with separate entities for each context"""
    import copy, uuid
    
    # Initialize graph structure
    graph = {"@graph": []}
    contexts = data.get("@context", [])
    if not isinstance(contexts, list): contexts = [contexts]
    
    # Create normalized root entity
    root = self._normalize_entity(copy.deepcopy(data))
    root_id = root.get("@id") or f"urn:uuid:{uuid.uuid4()}"
    root["@id"] = root_id
    
    # Assign first context to root entity
    if contexts and len(contexts) > 0:
        root["@context"] = contexts[0]
    
    # Keep track of processed entities
    processed_ids = set()
    
    # Process an entity and its nested objects
    def process_entity(entity, ctx_index=0):
        if not isinstance(entity, dict): return entity
        
        # Ensure ID
        entity_id = entity.get("@id")
        if not entity_id:
            entity_id = f"urn:uuid:{uuid.uuid4()}"
            entity["@id"] = entity_id
            
        # Skip if already processed
        if entity_id in processed_ids: return {"@id": entity_id}
        processed_ids.add(entity_id)
        
        # Assign context (alternate contexts for nested entities)
        if len(contexts) > 0:
            entity["@context"] = contexts[ctx_index % len(contexts)]
        
        # Create a copy for the graph
        graph_entity = copy.deepcopy(entity)
        
        # Process nested properties
        for prop, value in list(entity.items()):
            if prop.startswith("@"): continue
            
            # Process dict value
            if isinstance(value, dict) and len(value) > 1:
                nested = process_entity(value, ctx_index + 1)
                graph_entity[prop] = {"@id": nested["@id"]}
            
            # Process list of values
            elif isinstance(value, list):
                for i, item in enumerate(value):
                    if isinstance(item, dict) and len(item) > 1:
                        nested = process_entity(item, ctx_index + 1)
                        if isinstance(graph_entity[prop], list):
                            graph_entity[prop][i] = {"@id": nested["@id"]}
        
        # Add to graph
        graph["@graph"].append(graph_entity)
        return {"@id": entity_id}
    
    # Start processing from root
    process_entity(root)
    
    return graph


## Context and Entity Normalization

In [ ]:
#| export
@patch
def _normalize_entity(self:SemanticMemory, data):
    """Normalize entity properties for consistent processing"""
    if not isinstance(data, dict): return data
    
    result = data.copy()
    
    # Normalize id → @id
    if "id" in result:
        if "@id" not in result: result["@id"] = result["id"]
        del result["id"]
    
    # Normalize type → @type
    if "type" in result:
        if "@type" not in result: result["@type"] = result["type"]
        del result["type"]
    
    # Recursively normalize nested objects
    for k, v in list(result.items()):
        if k.startswith("@"): continue
        if isinstance(v, dict): result[k] = self._normalize_entity(v)
        elif isinstance(v, list): result[k] = [self._normalize_entity(i) if isinstance(i, dict) else i for i in v]
    
    return result

In [ ]:
#| export
@patch
def _register_context(self:SemanticMemory, context):
    """Register context and extract scoped contexts"""
    import json, hashlib
    
    # Generate stable ID for context
    if isinstance(context, str): ctx_id = context
    elif isinstance(context, list): ctx_id = "ctx:" + "_".join(self._register_context(c) for c in context)
    elif isinstance(context, dict):
        ctx_str = json.dumps(context, sort_keys=True)
        ctx_id = "ctx:" + hashlib.md5(ctx_str.encode()).hexdigest()
    else: return None
    
    # Skip if already registered
    if ctx_id in self.context_registry: return ctx_id
    
    # Register new context
    self.context_registry[ctx_id] = {
        "context": context,
        "scoped_contexts": {},
        "used_by": set()
    }
    
    # Extract scoped contexts if present
    if isinstance(context, dict):
        for term, defn in context.items():
            if isinstance(defn, dict) and "@context" in defn:
                # Register the scoped context
                scoped_id = self._register_context(defn["@context"])
                if scoped_id: self.context_registry[ctx_id]["scoped_contexts"][term] = scoped_id
    
    return ctx_id


## Quick Tests

In [ ]:
def test_graph_partitioning():
    memory = SemanticMemory()
    
    # Test detection of need for partitioning
    vc_data = {
        "@context": [
            "https://www.w3.org/ns/credentials/v2",
            "https://schema.org/"
        ],
        "id": "urn:test:vc:1",
        "type": ["VerifiableCredential"],
        "credentialSubject": {
            "id": "urn:test:subject:1",
            "type": "Person",
            "name": "Test Subject"
        }
    }
    
    # Test detection
    needs_partition = memory._needs_graph_partition(vc_data)
    assert needs_partition, "Failed to detect need for partitioning"
    
    # Test partitioning
    graph_data = memory._create_graph_partition(vc_data)
    assert "@graph" in graph_data, "Graph structure not created"
    assert len(graph_data["@graph"]) == 2, f"Expected 2 entities, got {len(graph_data['@graph'])}"
    
    # Verify entities in graph
    entities = graph_data["@graph"]
    
    # Find credential and subject
    credential = next((e for e in entities if "@type" in e and "VerifiableCredential" in e["@type"]), None)
    subject = next((e for e in entities if e.get("@id") == "urn:test:subject:1"), None)
    
    assert credential is not None, "Credential not found in graph"
    assert subject is not None, "Subject not found in graph"
    
    # Check reference
    assert "credentialSubject" in credential
    assert "@id" in credential["credentialSubject"]
    assert credential["credentialSubject"]["@id"] == subject["@id"]
    
    # Check contexts
    assert "@context" in credential
    assert "@context" in subject
    assert credential["@context"] != subject["@context"]
    
    print("✓ Graph partitioning tests passed")
    return graph_data


In [ ]:
gd = test_graph_partitioning()

Error preloading vocabulary schema: name 'load_context_for_vocabulary' is not defined
Error preloading vocabulary dc: name 'load_context_for_vocabulary' is not defined
Error preloading vocabulary foaf: name 'load_context_for_vocabulary' is not defined
✓ Graph partitioning tests passed


In [ ]:
def test_register_context():
    memory = SemanticMemory()
    
    # Test simple string context
    ctx_id1 = memory._register_context("https://schema.org/")
    assert ctx_id1 == "https://schema.org/"
    assert ctx_id1 in memory.context_registry
    
    # Test object context
    obj_context = {
        "@vocab": "https://schema.org/",
        "name": "https://schema.org/name"
    }
    ctx_id2 = memory._register_context(obj_context)
    assert ctx_id2.startswith("ctx:")
    assert ctx_id2 in memory.context_registry
    assert memory.context_registry[ctx_id2]["context"] == obj_context
    
    # Test scoped context
    scoped_context = {
        "@vocab": "https://schema.org/",
        "nested": {
            "@id": "https://example.org/nested",
            "@context": {
                "@vocab": "https://example.org/"
            }
        }
    }
    ctx_id3 = memory._register_context(scoped_context)
    assert ctx_id3 in memory.context_registry
    assert "nested" in memory.context_registry[ctx_id3]["scoped_contexts"]
    
    # Test list context
    list_context = ["https://schema.org/", "https://www.w3.org/ns/activitystreams"]
    ctx_id4 = memory._register_context(list_context)
    assert ctx_id4.startswith("ctx:")
    assert ctx_id4 in memory.context_registry
    
    print("✓ Context registration tests passed")
    return memory.context_registry


In [ ]:
test_register_context()

Error preloading vocabulary schema: name 'load_context_for_vocabulary' is not defined
Error preloading vocabulary dc: name 'load_context_for_vocabulary' is not defined
Error preloading vocabulary foaf: name 'load_context_for_vocabulary' is not defined
✓ Context registration tests passed


{'https://schema.org/': {'context': 'https://schema.org/',
  'scoped_contexts': {},
  'used_by': set()},
 'ctx:2dad20c90d4504209dee161dcff07e9c': {'context': {'@vocab': 'https://schema.org/',
   'name': 'https://schema.org/name'},
  'scoped_contexts': {},
  'used_by': set()},
 'ctx:2f71d18305da5105504cc67dba26a70c': {'context': {'@vocab': 'https://schema.org/',
   'nested': {'@id': 'https://example.org/nested',
    '@context': {'@vocab': 'https://example.org/'}}},
  'scoped_contexts': {'nested': 'ctx:f7582947ddd802a865250288ac9d23e1'},
  'used_by': set()},
 'ctx:f7582947ddd802a865250288ac9d23e1': {'context': {'@vocab': 'https://example.org/'},
  'scoped_contexts': {},
  'used_by': set()},
 'https://www.w3.org/ns/activitystreams': {'context': 'https://www.w3.org/ns/activitystreams',
  'scoped_contexts': {},
  'used_by': set()},
 'ctx:https://schema.org/_https://www.w3.org/ns/activitystreams': {'context': ['https://schema.org/',
   'https://www.w3.org/ns/activitystreams'],
  'scoped_cont

In [ ]:
def test_normalize_entity():
    memory = SemanticMemory()
    
    # Test basic normalization
    test_entity = {
        "id": "urn:test:entity:1",
        "type": "Person",
        "name": "Test Person"
    }
    
    normalized = memory._normalize_entity(test_entity)
    
    # Check id/type normalization
    assert "@id" in normalized and "id" not in normalized
    assert "@type" in normalized and "type" not in normalized
    assert normalized["@id"] == "urn:test:entity:1"
    assert normalized["@type"] == "Person"
    
    # Test nested normalization
    nested_entity = {
        "id": "urn:test:entity:2",
        "type": "Organization",
        "name": "Test Org",
        "contact": {
            "id": "urn:test:contact:1",
            "type": "ContactPoint",
            "email": "test@example.com"
        }
    }
    
    normalized_nested = memory._normalize_entity(nested_entity)
    
    # Check nested normalization
    assert "@id" in normalized_nested["contact"]
    assert "id" not in normalized_nested["contact"]
    assert "@type" in normalized_nested["contact"]
    assert "type" not in normalized_nested["contact"]
    
    print("✓ Entity normalization tests passed")
    return normalized_nested


In [ ]:
norm = test_normalize_entity()

Error preloading vocabulary schema: name 'load_context_for_vocabulary' is not defined
Error preloading vocabulary dc: name 'load_context_for_vocabulary' is not defined
Error preloading vocabulary foaf: name 'load_context_for_vocabulary' is not defined
✓ Entity normalization tests passed


## Querying Memory

In [ ]:
#| export
@patch
def query_by_id(self:SemanticMemory, entity_id):
    """Get entity by ID with improved retrieval"""
    # Try direct lookup
    if entity_id in self.indices["by_id"]: return self.indices["by_id"][entity_id]
    
    # Try with/without trailing slash for URLs
    if entity_id.startswith(('http://', 'https://')):
        alt_id = entity_id + '/' if not entity_id.endswith('/') else entity_id[:-1]
        if alt_id in self.indices["by_id"]: return self.indices["by_id"][alt_id]
    
    # Try RDF query as fallback
    entity_uri = URIRef(entity_id)
    results = {}
    
    for s, p, o in self.default_graph.triples((entity_uri, None, None)):
        pred = str(p)
        if pred not in results: results[pred] = []
        
        if isinstance(o, URIRef): results[pred].append({"@id": str(o)})
        elif isinstance(o, Literal): results[pred].append({"@value": str(o)})
    
    return results if results else None

In [ ]:
#| export
@patch
def get_connected_entities(self:SemanticMemory, entity_id, max_depth=1):
    """Get entity and all connected entities up to max_depth"""
    if max_depth < 0: return {}
    
    # Get the starting entity
    entity = self.query_by_id(entity_id)
    if not entity: return {}
    
    result = {entity_id: entity}
    
    # Find all referenced entities
    if max_depth > 0:
        for prop, values in entity.items():
            if prop.startswith("@"): continue
            
            for val in values if isinstance(values, list) else [values]:
                if isinstance(val, dict) and "@id" in val:
                    ref_id = val["@id"]
                    # Recursively get connected entities
                    connected = self.get_connected_entities(ref_id, max_depth-1)
                    result.update(connected)
    
    return result

In [ ]:
#| export
@patch 
def query_by_type(self:SemanticMemory, type_uri):
    """Get all entities of a specific type, including subtypes"""
    results = []
    seen_ids = set()  # Track entities we've already added
    processed_types = set()  # Track types we've already processed
    
    # Normalize URIs to handle schema.org HTTP/HTTPS variations
    uris_to_check = [type_uri]
    if type_uri.startswith("http://schema.org/"):
        uris_to_check.append(type_uri.replace("http://", "https://"))
    elif type_uri.startswith("https://schema.org/"):
        uris_to_check.append(type_uri.replace("https://", "http://"))
    
    # Helper to add entities of a specific type to results
    def add_entities_of_type(t):
        if t in self.indices["by_type"]:
            for entity in self.indices["by_type"][t]:
                entity_id = entity.get("@id")
                if entity_id and entity_id not in seen_ids:
                    results.append(entity)
                    seen_ids.add(entity_id)
    
    # Helper to process a type and its subtypes using iteration instead of recursion
    def process_type_hierarchy(start_type):
        to_process = [start_type]  # Stack of types to process
        
        while to_process:
            current_type = to_process.pop()
            
            # Skip if already processed
            if current_type in processed_types:
                continue
                
            processed_types.add(current_type)
            
            # Add entities of this type
            add_entities_of_type(current_type)
            
            # Add subtypes to processing stack
            if "superclass_of" in self.indices:
                for alt_uri in [current_type]:
                    if alt_uri in self.indices["superclass_of"]:
                        for subtype in self.indices["superclass_of"][alt_uri]:
                            if subtype not in processed_types:
                                to_process.append(subtype)
    
    # Process all URIs we need to check
    for uri in uris_to_check:
        if uri not in processed_types:
            process_type_hierarchy(uri)
    
    # If no results found, try RDF graph query as fallback
    if not results:
        try:
            for uri in uris_to_check:
                type_uri_ref = URIRef(uri)
                for s in self.default_graph.subjects(RDF.type, type_uri_ref):
                    entity_id = str(s)
                    if entity_id not in seen_ids:
                        entity = self.query_by_id(entity_id)
                        if entity:
                            results.append(entity)
                            seen_ids.add(entity_id)
        except Exception as e:
            print(f"RDF query error: {e}")
    
    return results


## Update Indexing

In [ ]:
#| export
@patch
def _update_indices(self:SemanticMemory, expanded_data):
    """Update indices with expanded JSON-LD data"""
    # Handle both single entity and array cases
    nodes = expanded_data if isinstance(expanded_data, list) else [expanded_data]
    
    for node in nodes:
        if not isinstance(node, dict): continue
        
        # Get node ID
        node_id = node.get('@id')
        if not node_id: continue
        
        # Update by_id index
        self.indices["by_id"][node_id] = node
        
        # Update by_type index - handle expanded types properly
        types = []
        
        # Check for @type property (could be string, list, or in expanded form)
        if '@type' in node:
            type_values = node['@type'] if isinstance(node['@type'], list) else [node['@type']]
            for t in type_values:
                if isinstance(t, str): types.append(t)
        
        # Also check for rdf:type triples in expanded JSON-LD
        rdf_type = 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type'
        if rdf_type in node:
            type_objects = node[rdf_type] if isinstance(node[rdf_type], list) else [node[rdf_type]]
            for t_obj in type_objects:
                if isinstance(t_obj, dict) and '@id' in t_obj:
                    types.append(t_obj['@id'])
        
        # Index by each type
        for type_uri in types:
            if type_uri not in self.indices["by_type"]: self.indices["by_type"][type_uri] = []
            if node not in self.indices["by_type"][type_uri]: 
                self.indices["by_type"][type_uri].append(node)
        
        # Update label and description indices
        self._index_labels_and_descriptions(node)
        
        # Track subClassOf relationships for type inference
        self._index_subclass_relationships(node)


In [ ]:
#| export
@patch
def _index_labels_and_descriptions(self:SemanticMemory, node):
    """Index entity by labels and textual content for better search"""
    if not isinstance(node, dict): return
    
    # Common label properties
    label_props = [
        "http://www.w3.org/2000/01/rdf-schema#label",
        "http://schema.org/name",
        "http://purl.org/dc/terms/title",
        "http://xmlns.com/foaf/0.1/name",
        "name"  # Handle both expanded and compact forms
    ]
    
    # Common description properties
    desc_props = [
        "http://www.w3.org/2000/01/rdf-schema#comment",
        "http://schema.org/description",
        "http://purl.org/dc/terms/description",
        "http://purl.org/dc/elements/1.1/description",
        "description"  # Handle both expanded and compact forms
    ]
    
    node_id = node.get('@id')
    if not node_id: return
    
    # Ensure index structures exist
    if "by_label" not in self.indices: self.indices["by_label"] = {}
    if "by_keyword" not in self.indices: self.indices["by_keyword"] = {}
    
    # Extract labels for label index
    for prop in label_props:
        if prop in node:
            values = node[prop] if isinstance(node[prop], list) else [node[prop]]
            for value in values:
                label = value.get('@value') if isinstance(value, dict) else value
                if isinstance(label, str):
                    # Add to label index
                    if label not in self.indices["by_label"]: self.indices["by_label"][label] = []
                    if {'@id': node_id} not in self.indices["by_label"][label]:
                        self.indices["by_label"][label].append({'@id': node_id})
                    
                    # Index label as a whole
                    label_lower = label.lower()
                    if label_lower not in self.indices["by_keyword"]: self.indices["by_keyword"][label_lower] = []
                    if {'@id': node_id} not in self.indices["by_keyword"][label_lower]:
                        self.indices["by_keyword"][label_lower].append({'@id': node_id})
                    
                    # Index individual words and partial words
                    for word in label.split():
                        word_lower = word.lower()
                        if len(word_lower) > 2:
                            # Add whole word
                            if word_lower not in self.indices["by_keyword"]: self.indices["by_keyword"][word_lower] = []
                            if {'@id': node_id} not in self.indices["by_keyword"][word_lower]:
                                self.indices["by_keyword"][word_lower].append({'@id': node_id})
                            
                            # Add partial words (prefixes)
                            for i in range(3, len(word_lower)+1):
                                prefix = word_lower[:i]
                                if prefix not in self.indices["by_keyword"]: self.indices["by_keyword"][prefix] = []
                                if {'@id': node_id} not in self.indices["by_keyword"][prefix]:
                                    self.indices["by_keyword"][prefix].append({'@id': node_id})
    
    # Extract text from descriptions for keyword index
    for prop in desc_props:
        if prop in node:
            values = node[prop] if isinstance(node[prop], list) else [node[prop]]
            for value in values:
                text = value.get('@value') if isinstance(value, dict) else value
                if isinstance(text, str):
                    # Simple keyword extraction - split and filter
                    for word in text.split():
                        word_lower = word.lower()
                        if len(word_lower) > 2:
                            if word_lower not in self.indices["by_keyword"]: self.indices["by_keyword"][word_lower] = []
                            if {'@id': node_id} not in self.indices["by_keyword"][word_lower]:
                                self.indices["by_keyword"][word_lower].append({'@id': node_id})


In [ ]:
#|export
@patch
def _index_subclass_relationships(self:SemanticMemory, node):
    """Index subclass relationships for type inference"""
    if not isinstance(node, dict): return
    
    # Initialize subclass index if needed
    if "subclass_of" not in self.indices: self.indices["subclass_of"] = {}
    if "superclass_of" not in self.indices: self.indices["superclass_of"] = {}
    
    # Check for subClassOf relationship
    subclass_of = 'http://www.w3.org/2000/01/rdf-schema#subClassOf'
    if subclass_of in node:
        class_id = node.get('@id')
        if not class_id: return
        
        # Get all superclasses
        super_classes = node[subclass_of] if isinstance(node[subclass_of], list) else [node[subclass_of]]
        
        for super_class in super_classes:
            if isinstance(super_class, dict) and '@id' in super_class:
                super_class_id = super_class['@id']
                
                # Record subclass relationship
                if class_id not in self.indices["subclass_of"]: 
                    self.indices["subclass_of"][class_id] = set()
                self.indices["subclass_of"][class_id].add(super_class_id)
                
                # Record superclass relationship (inverse)
                if super_class_id not in self.indices["superclass_of"]:
                    self.indices["superclass_of"][super_class_id] = set()
                self.indices["superclass_of"][super_class_id].add(class_id)


## Retrieval and Search Functions

In [ ]:
#| export
@patch
def retrieve_in_context(self:SemanticMemory, entity_id):
    """Retrieve entity with original context structure preserved"""
    # Check if we have the original data
    if entity_id not in self.original_data: return None
    
    # Get the original data with context
    entity = self.original_data[entity_id]
    
    # Look up context registry entries for this entity
    ctx_ids = []
    for ctx_id, ctx_info in self.context_registry.items():
        if entity_id in ctx_info.get("used_by", set()):
            ctx_ids.append(ctx_id)
    
    # Ensure scoped contexts are properly included
    import copy
    result = copy.deepcopy(entity)
    
    # Process any context with scoped contexts
    for ctx_id in ctx_ids:
        ctx_info = self.context_registry[ctx_id]
        
        # Check if this context has scoped contexts
        if ctx_info["scoped_contexts"] and isinstance(result.get("@context"), dict):
            # Ensure all scoped contexts are included
            for term, scoped_id in ctx_info["scoped_contexts"].items():
                if term in result["@context"] and isinstance(result["@context"][term], dict):
                    # Get the scoped context definition
                    scoped_ctx = self.context_registry.get(scoped_id, {}).get("context")
                    if scoped_ctx:
                        # Add back the scoped context
                        result["@context"][term]["@context"] = scoped_ctx
    
    return result


In [ ]:
#| export
@patch
def search(self:SemanticMemory, query_text, limit=10):
    """Search memory for entities matching the query text"""
    results = []
    query_lower = query_text.lower()
    
    # Track matched entities to avoid duplicates
    matched_ids = set()
    
    # First check exact label matches (highest priority)
    if "by_label" in self.indices:
        for label, entities in self.indices["by_label"].items():
            if isinstance(label, str) and query_lower in label.lower():
                for entity_ref in entities:
                    if "@id" in entity_ref:
                        entity_id = entity_ref["@id"]
                        if entity_id not in matched_ids:
                            entity = self.query_by_id(entity_id)
                            if entity:
                                results.append(entity)
                                matched_ids.add(entity_id)
                                if len(results) >= limit: return results
    
    # Then check keyword matches
    if "by_keyword" in self.indices:
        # Try the exact token first
        if query_lower in self.indices["by_keyword"]:
            for entity_ref in self.indices["by_keyword"][query_lower]:
                if "@id" in entity_ref:
                    entity_id = entity_ref["@id"]
                    if entity_id not in matched_ids:
                        entity = self.query_by_id(entity_id)
                        if entity:
                            results.append(entity)
                            matched_ids.add(entity_id)
                            if len(results) >= limit: return results
        
        # Try as partial match for any keywords
        for keyword, entities in self.indices["by_keyword"].items():
            if isinstance(keyword, str) and query_lower in keyword and query_lower != keyword:
                for entity_ref in entities:
                    if "@id" in entity_ref:
                        entity_id = entity_ref["@id"]
                        if entity_id not in matched_ids:
                            entity = self.query_by_id(entity_id)
                            if entity:
                                results.append(entity)
                                matched_ids.add(entity_id)
                                if len(results) >= limit: return results
    
    # If still not enough results, try content search
    if len(results) < limit:
        for entity_id, entity in self.indices["by_id"].items():
            if entity_id in matched_ids: continue
            
            # Try to find the query in string representation of values
            for prop, values in entity.items():
                if prop.startswith('@'): continue
                
                # Handle different value formats
                if isinstance(values, list):
                    for val in values:
                        # Convert to string safely
                        val_str = ""
                        if isinstance(val, dict) and "@value" in val:
                            val_content = val.get("@value")
                            if isinstance(val_content, str):
                                val_str = val_content.lower()
                        elif isinstance(val, str):
                            val_str = val.lower()
                        
                        if val_str and query_lower in val_str:
                            results.append(entity)
                            matched_ids.add(entity_id)
                            break
                else:
                    # Convert to string safely
                    val_str = ""
                    if isinstance(values, dict) and "@value" in values:
                        val_content = values.get("@value")
                        if isinstance(val_content, str):
                            val_str = val_content.lower()
                    elif isinstance(values, str):
                        val_str = values.lower()
                    
                    if val_str and query_lower in val_str:
                        results.append(entity)
                        matched_ids.add(entity_id)
                
                if entity_id in matched_ids and len(results) >= limit: 
                    break
            
            if len(results) >= limit:
                break
    
    return results


In [ ]:
@patch
def semantic_search(self:SemanticMemory, q:str, limit:int=10, include_connected:bool=False, max_depth:int=1):
    print(f"Searching for: {q}")
    if q in self.indices["by_id"]: print("ID branch"); res=[self.indices["by_id"][q]]
    elif q.startswith(("http://","https://","urn:")): print("URL branch"); ent=self.query_by_id(q); res=[ent] if ent else []
    else:
        ts=self.indices.get("by_type",{}); b={t.rsplit('/',1)[-1] for t in ts}
        if q in ts or q in b: print("Type branch"); res=self.query_by_type(q)
        else: print("Keyword branch"); res=self.search(q,limit)
    if include_connected and res:
        d={}
        for e in res:
            eid=e.get("@id")
            if eid: d[eid]=e; d.update(self.get_connected_entities(eid, max_depth))
        res=list(d.values())
    print(f"Returning {len(res)} results")
    return res[:limit]

### LLM Tool Hooks for Semantic Memory

In [ ]:
#| export
@patch 
def query_by_type(self:SemanticMemory, type_uri):
    """Get all entities of a specific type, including subtypes"""
    results = []
    seen_ids = set()
    
    # Handle HTTP/HTTPS variations for any vocabulary
    types_to_check = set([type_uri])
    if type_uri.startswith("http://"):
        types_to_check.add(type_uri.replace("http://", "https://"))
    elif type_uri.startswith("https://"):
        types_to_check.add(type_uri.replace("https://", "http://"))
    
    # Add all subtypes recursively
    if "superclass_of" in self.indices:
        for t in list(types_to_check):  # Use list to avoid modifying during iteration
            if t in self.indices["superclass_of"]:
                types_to_check.update(self.indices["superclass_of"][t])
    
    # Get all entities of these types
    for t in types_to_check:
        if t in self.indices["by_type"]:
            for entity in self.indices["by_type"][t]:
                eid = entity.get("@id")
                if eid and eid not in seen_ids:
                    results.append(entity)
                    seen_ids.add(eid)
    
    return results


In [ ]:
#| export
@patch
def _get_connected_subgraph(self:SemanticMemory, entity_id, max_depth=1, visited=None):
    """Extract a semantically connected subgraph centered on an entity.
    
    This function traverses the knowledge graph starting from a central entity,
    following relationships to connected entities up to max_depth hops away.
    It maintains the graph structure and relationship semantics.
    
    Args:
        entity_id: The ID of the central entity
        max_depth: Maximum number of relationship hops to traverse
        visited: Set of already visited entity IDs (for recursion)
        
    Returns:
        Dictionary mapping entity IDs to their data for the subgraph
    """
    if max_depth < 0: return {}
    if visited is None: visited = set()
    
    # Skip if already visited
    if entity_id in visited: return {}
    
    # Mark as visited
    visited.add(entity_id)
    
    # Get the entity
    entity = self.query_by_id(entity_id)
    if not entity: return {}
    
    result = {entity_id: entity}
    
    # If at max depth, don't explore further
    if max_depth == 0: return result
    
    # Find all referenced entities
    for prop, values in entity.items():
        if prop.startswith("@"): continue
        
        # Handle list values
        if isinstance(values, list):
            for val in values:
                if isinstance(val, dict) and "@id" in val:
                    ref_id = val["@id"]
                    # Recursively get connected entities
                    connected = self._get_connected_subgraph(ref_id, max_depth-1, visited)
                    result.update(connected)
        # Handle dict values
        elif isinstance(values, dict) and "@id" in values:
            ref_id = values["@id"]
            # Recursively get connected entities
            connected = self._get_connected_subgraph(ref_id, max_depth-1, visited)
            result.update(connected)
    
    return result


In [ ]:
#| export
@patch
def prepare_for_llm(self:SemanticMemory, entities, include_vocab_info=True, compact=True):
    """Format entities for optimal LLM understanding and context window usage.
    
    This function processes a set of entities to make them more suitable for
    loading into an LLM's context window. It can:
    1. Detect and include vocabulary information
    2. Compact entities using their appropriate vocabularies
    3. Extract explicit relationship information
    
    Args:
        entities: List of entities to prepare
        include_vocab_info: Whether to include vocabulary metadata
        compact: Whether to compact entities with their vocabularies
        
    Returns:
        Structured dictionary with entities, relationships, and vocabulary info
    """
    if not entities: return {}
    
    # Detect vocabularies used
    vocab_info = {}
    if include_vocab_info:
        for entity in entities:
            if isinstance(entity, dict) and "@context" in entity:
                ctx = entity["@context"]
                # Process string contexts
                if isinstance(ctx, str):
                    for vocab_name, vocab_data in VOCABULARY_REGISTRY.items():
                        if vocab_data["uri"] in ctx:
                            vocab_info[vocab_name] = vocab_data["uri"]
                # Process object contexts
                elif isinstance(ctx, dict) and "@vocab" in ctx:
                    vocab_uri = ctx["@vocab"]
                    for vocab_name, vocab_data in VOCABULARY_REGISTRY.items():
                        if vocab_data["uri"] == vocab_uri:
                            vocab_info[vocab_name] = vocab_data["uri"]
    
    # Compact entities if requested
    processed_entities = []
    for entity in entities:
        if compact and isinstance(entity, dict):
            # Try to detect vocabulary
            detected_vocab = None
            for vocab_name in vocab_info:
                if isinstance(entity.get("@context"), str) and vocab_info[vocab_name] in entity["@context"]:
                    detected_vocab = vocab_name
                    break
            
            # Compact if vocabulary detected
            if detected_vocab:
                try:
                    compacted = compact_entity_with_vocabulary(entity, detected_vocab)
                    processed_entities.append(compacted)
                except:
                    processed_entities.append(entity)
            else:
                processed_entities.append(entity)
        else:
            processed_entities.append(entity)
    
    # Create result structure
    result = {
        "entities": processed_entities,
        "count": len(processed_entities),
        "relationships": self._extract_relationships(processed_entities)
    }
    
    # Add vocabulary information if available
    if vocab_info:
        result["vocabularies"] = vocab_info
    
    return result


In [ ]:
#| export
@patch
def _extract_relationships(self:SemanticMemory, entities):
    """Extract explicit relationships between entities for LLM comprehension.
    
    This function analyzes a set of entities and identifies all the relationships
    between them, making the knowledge graph structure explicit and easier for
    an LLM to understand.
    
    Args:
        entities: List of entities to analyze
        
    Returns:
        List of relationship objects with source, property, and target
    """
    if not entities: return []
    
    relationships = []
    entity_ids = {e.get("@id") for e in entities if isinstance(e, dict) and "@id" in e}
    
    for entity in entities:
        if not isinstance(entity, dict) or "@id" not in entity: continue
        
        source_id = entity["@id"]
        
        # Find all references to other entities in the set
        for prop, values in entity.items():
            if prop.startswith("@"): continue
            
            # Handle list values
            if isinstance(values, list):
                for val in values:
                    if isinstance(val, dict) and "@id" in val:
                        target_id = val["@id"]
                        if target_id in entity_ids:
                            relationships.append({
                                "source": source_id,
                                "property": prop,
                                "target": target_id
                            })
            # Handle dict values
            elif isinstance(values, dict) and "@id" in values:
                target_id = values["@id"]
                if target_id in entity_ids:
                    relationships.append({
                        "source": source_id,
                        "property": prop,
                        "target": target_id
                    })
    
    return relationships


In [ ]:
#| export
@patch
def search_for_llm(self:SemanticMemory, query, limit=10, include_connected=True, max_depth=1, include_vocab_info=True):
    """Search semantic memory and prepare results specifically for LLM consumption.
    
    This convenience function combines semantic search with LLM-optimized formatting
    to make it easy to load relevant knowledge graph subsets into an LLM context window.
    
    Args:
        query: Text to search for (can be an ID, type, label, or keywords)
        limit: Maximum number of entities to return
        include_connected: Whether to include connected entities in results
        max_depth: How many relationship hops to traverse
        include_vocab_info: Whether to include vocabulary metadata
        
    Returns:
        Structured data ready for loading into an LLM context window
    """
    entities = self.semantic_search(query, limit, include_connected, max_depth)
    return self.prepare_for_llm(entities, include_vocab_info)


In [ ]:
#| export
@patch
def prepare_for_llm_context(self:SemanticMemory, query=None, graph_id=None, limit=50, preferred_vocabs=None):
    "Prepare memory contents for LLM context window with vocabulary awareness"
    entities = []
    
    if query:
        search_results = self.search(query, limit=limit)
        entities.extend(search_results)
    elif graph_id:
        if "by_graph" in self.indices and URIRef(graph_id) in self.indices["by_graph"]:
            entity_ids = list(self.indices["by_graph"][URIRef(graph_id)])[:limit]
            for eid in entity_ids:
                entity = self.query_by_id(eid)
                if entity: entities.append(entity)
    else:
        entity_ids = list(self.indices["by_id"].keys())[:limit]
        for eid in entity_ids:
            entity = self.query_by_id(eid)
            if entity: entities.append(entity)
    
    formatted_entities = []
    for entity in entities:
        compacted = self.compact_with_preferred_vocab(entity, preferred_vocabs)
        formatted_entities.appen


## Tests

In [ ]:
def test_graph_partitioning_contexts():
    memory = SemanticMemory()
    
    # Create test data with multiple contexts and nested objects
    test_data = {
        "@context": [
            "https://schema.org/",
            "https://www.w3.org/ns/activitystreams"
        ],
        "id": "urn:test:root",
        "type": "Organization",
        "name": "Test Organization",
        "department": {
            "id": "urn:test:dept1",
            "type": "Department",
            "name": "Department 1",
            "employee": {
                "id": "urn:test:person1",
                "type": "Person",
                "name": "Person 1"
            }
        },
        "projects": [
            {
                "id": "urn:test:project1",
                "type": "Project",
                "name": "Project 1"
            },
            {
                "id": "urn:test:project2",
                "type": "Project",
                "name": "Project 2",
                "team": {
                    "id": "urn:test:team1",
                    "type": "Group",
                    "name": "Team 1"
                }
            }
        ]
    }
    
    # Run the partitioning
    graph_data = memory._create_graph_partition(test_data)
    
    # Verify the basic structure
    assert "@graph" in graph_data, "Graph structure not created"
    entities = graph_data["@graph"]
    
    # Count entities (should be 6: root, dept, person, 2 projects, team)
    expected_count = 6
    assert len(entities) == expected_count, f"Expected {expected_count} entities, got {len(entities)}"
    
    # Map entities by ID for easier access
    entity_map = {e["@id"]: e for e in entities}
    
    # Verify all expected IDs are present
    expected_ids = ["urn:test:root", "urn:test:dept1", "urn:test:person1", 
                   "urn:test:project1", "urn:test:project2", "urn:test:team1"]
    for eid in expected_ids:
        assert eid in entity_map, f"Entity {eid} missing from graph"
    
    # Check context assignments
    contexts = test_data["@context"]
    if len(contexts) > 1:
        # Root should have first context
        assert entity_map["urn:test:root"]["@context"] == contexts[0]
        
        # Check department (direct child of root - should have different context than root)
        dept_ctx = entity_map["urn:test:dept1"]["@context"]
        assert dept_ctx != entity_map["urn:test:root"]["@context"], "Department has same context as root"
        
        # Check person (child of department - should have different context than department)
        person_ctx = entity_map["urn:test:person1"]["@context"]
        assert person_ctx != dept_ctx, "Person has same context as department"
    
    # Verify references are maintained
    root = entity_map["urn:test:root"]
    assert "@id" in root["department"], "Department reference lost"
    assert root["department"]["@id"] == "urn:test:dept1"
    
    # Verify project references
    assert isinstance(root["projects"], list), "Projects should be a list"
    assert len(root["projects"]) == 2, "Should have 2 project references"
    assert root["projects"][0]["@id"] == "urn:test:project1"
    assert root["projects"][1]["@id"] == "urn:test:project2"
    
    # Verify deeper nesting
    dept = entity_map["urn:test:dept1"]
    assert "@id" in dept["employee"], "Employee reference lost"
    assert dept["employee"]["@id"] == "urn:test:person1"
    
    project2 = entity_map["urn:test:project2"]
    assert "@id" in project2["team"], "Team reference lost"
    assert project2["team"]["@id"] == "urn:test:team1"
    
    print("✓ Graph partitioning with context assignment passed")
    return entity_map


In [ ]:
em = test_graph_partitioning_contexts()

Error preloading vocabulary schema: name 'load_context_for_vocabulary' is not defined
Error preloading vocabulary dc: name 'load_context_for_vocabulary' is not defined
Error preloading vocabulary foaf: name 'load_context_for_vocabulary' is not defined
✓ Graph partitioning with context assignment passed


In [ ]:
def test_add_function():    
    # Register the vocabulary-aware document loader
    register_vocab_aware_loader()
    
    memory = SemanticMemory()
    
    # Test adding a simple entity
    simple_entity = {
        "@context": "https://schema.org/",
        "id": "urn:test:simple:1",
        "type": "Person",
        "name": "Simple Entity"
    }
    
    result = memory.add(simple_entity)
    assert result is not None, "Add returned None"
    
    # Verify it was added to indices
    entity = memory.query_by_id("urn:test:simple:1")
    assert entity is not None, "Entity not found in indices"
    
    # Check if type index contains Person (with or without full URI)
    has_type = False
    for type_key in memory.indices["by_type"]:
        if "Person" in type_key:
            has_type = True
            break
    assert has_type, "Type index not updated with Person type"
    
    # Test adding an entity that needs partitioning
    complex_entity = {
        "@context": [
            "https://schema.org/",
            "https://www.w3.org/ns/activitystreams"
        ],
        "id": "urn:test:complex:1",
        "type": ["Person", "Actor"],
        "name": "Complex Entity",
        "outbox": {
            "id": "urn:test:outbox:1",
            "type": "OrderedCollection",
            "totalItems": 10
        }
    }
    
    results = memory.add(complex_entity)
    assert results is not None, "Complex add failed"
    
    # Verify both entities were added
    entity1 = memory.query_by_id("urn:test:complex:1")
    entity2 = memory.query_by_id("urn:test:outbox:1")
    assert entity1 is not None, "Main entity not found"
    assert entity2 is not None, "Nested entity not found"
    
    print("✓ Add function tests passed")
    return memory


In [ ]:
test_add_function()

Error preloading vocabulary schema: name 'load_context_for_vocabulary' is not defined
Error preloading vocabulary dc: name 'load_context_for_vocabulary' is not defined
Error preloading vocabulary foaf: name 'load_context_for_vocabulary' is not defined


NameError: name 'detect_vocabularies_in_context' is not defined

In [ ]:
def test_query_and_traversal():
    memory = SemanticMemory()
    
    # Set up a network of related entities
    memory.add({
        "@context": "https://schema.org/",
        "id": "urn:test:person:a",
        "type": "Person",
        "name": "Person A",
        "knows": [
            {"id": "urn:test:person:b"},
            {"id": "urn:test:person:c"}
        ]
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "id": "urn:test:person:b",
        "type": "Person",
        "name": "Person B"
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "id": "urn:test:person:c",
        "type": "Person",
        "name": "Person C",
        "knows": {"id": "urn:test:person:d"}
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "id": "urn:test:person:d",
        "type": "Person",
        "name": "Person D"
    })
    
    # Test query_by_id
    person_a = memory.query_by_id("urn:test:person:a")
    assert person_a is not None, "Failed to retrieve Person A"
    
    # Test query_by_type
    persons = memory.query_by_type("http://schema.org/Person")
    assert len(persons) == 4, f"Expected 4 persons, got {len(persons)}"
    
    # Test relationship traversal
    connected = memory.get_connected_entities("urn:test:person:a", max_depth=2)
    assert len(connected) >= 3, f"Expected at least 3 connected entities, got {len(connected)}"
    assert "urn:test:person:d" in connected, "Failed to traverse to depth 2"
    
    print("✓ Query and traversal tests passed")
    return connected


In [ ]:
con = test_query_and_traversal()

✓ Query and traversal tests passed


In [ ]:
def test_search():
    memory = SemanticMemory()
    
    # Add some entities with searchable content
    memory.add({
        "@context": {"@vocab": "http://schema.org/"},
        "id": "urn:test:product:1",
        "type": "Product",
        "name": "Smartphone X200",
        "description": "Latest smartphone with advanced features"
    })
    
    memory.add({
        "@context": {"@vocab": "http://schema.org/"},
        "id": "urn:test:product:2",
        "type": "Product",
        "name": "Laptop Pro",
        "description": "Professional laptop for developers"
    })
    
    memory.add({
        "@context": {"@vocab": "http://schema.org/"},
        "id": "urn:test:organization:1",
        "type": "Organization",
        "name": "TechCorp",
        "description": "Manufacturer of smartphones and laptops"
    })
    
    # Test search by name
    results1 = memory.search("Smartphone")
    assert len(results1) >= 1, "Failed to find entity by name"
    assert any("urn:test:product:1" == e.get("@id") for e in results1), "Smartphone not found"
    
    # Test search by description keyword
    results2 = memory.search("developers")
    assert len(results2) >= 1, "Failed to find entity by description keyword"
    assert any("urn:test:product:2" == e.get("@id") for e in results2), "Laptop not found"
    
    # Test search that should match multiple entities
    results3 = memory.search("tech")
    assert len(results3) >= 2, "Failed to find multiple matching entities"
    
    print("✓ Search tests passed")
    return results3


In [ ]:
def test_search_debug():
    memory = SemanticMemory()
    
    # Add some entities with searchable content
    memory.add({
        "@context": {"@vocab": "http://schema.org/"},
        "id": "urn:test:product:1",
        "type": "Product",
        "name": "Smartphone X200",
        "description": "Latest smartphone with advanced features"
    })
    
    memory.add({
        "@context": {"@vocab": "http://schema.org/"},
        "id": "urn:test:product:2",
        "type": "Product",
        "name": "Laptop Pro",
        "description": "Professional laptop for developers"
    })
    
    memory.add({
        "@context": {"@vocab": "http://schema.org/"},
        "id": "urn:test:organization:1",
        "type": "Organization",
        "name": "TechCorp",
        "description": "Manufacturer of smartphones and laptops"
    })
    
    # Print the indices for debugging
    print("Label index:", list(memory.indices.get("by_label", {}).keys()))
    print("Keyword index:", list(memory.indices.get("by_keyword", {}).keys()))
    
    # Helper to safely extract name
    def get_name(entity):
        name = entity.get('name') or entity.get('http://schema.org/name')
        if isinstance(name, list): 
            return name[0].get('@value') if isinstance(name[0], dict) else name[0] if name else "No name"
        elif isinstance(name, dict):
            return name.get('@value', "No name")
        else:
            return name or "No name"
    
    # Test search by name
    results1 = memory.search("Smartphone")
    print(f"Smartphone search results: {len(results1)}")
    for r in results1:
        print(f"  - {r.get('@id')}: {get_name(r)}")
    
    # Test search by description keyword
    results2 = memory.search("developers")
    print(f"Developers search results: {len(results2)}")
    for r in results2:
        print(f"  - {r.get('@id')}: {get_name(r)}")
    
    # Test search that should match multiple entities
    results3 = memory.search("tech")
    print(f"Tech search results: {len(results3)}")
    for r in results3:
        print(f"  - {r.get('@id')}: {get_name(r)}")
    
    # Dump full entity info for "tech" search to debug
    print("\nFull entity info for tech search:")
    for i, r in enumerate(results3):
        print(f"Entity {i+1}:")
        for k, v in r.items():
            print(f"  {k}: {v}")
    
    return memory


In [ ]:
test_search_debug()

Label index: ['Smartphone X200', 'Laptop Pro', 'TechCorp']
Keyword index: ['smartphone x200', 'smartphone', 'sma', 'smar', 'smart', 'smartp', 'smartph', 'smartpho', 'smartphon', 'x200', 'x20', 'latest', 'with', 'advanced', 'features', 'laptop pro', 'laptop', 'lap', 'lapt', 'lapto', 'pro', 'professional', 'for', 'developers', 'techcorp', 'tec', 'tech', 'techc', 'techco', 'techcor', 'manufacturer', 'smartphones', 'and', 'laptops']
Smartphone search results: 2
  - urn:test:product:1: Smartphone X200
  - urn:test:organization:1: TechCorp
Developers search results: 1
  - urn:test:product:2: Laptop Pro
Tech search results: 1
  - urn:test:organization:1: TechCorp

Full entity info for tech search:
Entity 1:
  @id: urn:test:organization:1
  @type: ['http://schema.org/Organization']
  http://schema.org/description: [{'@value': 'Manufacturer of smartphones and laptops'}]
  http://schema.org/name: [{'@value': 'TechCorp'}]


<__main__.SemanticMemory>

In [ ]:
def test_semantic_search_and_llm_prep():
    """Test the semantic search and LLM preparation functionality"""
    
    # Register the vocabulary-aware document loader
    register_vocab_aware_loader()
    
    # Create a new memory instance
    memory = SemanticMemory()
    
    print("=== Testing Semantic Search and LLM Preparation ===\n")
    
    # 1. Create a small knowledge graph for testing
    print("1. Creating test knowledge graph...")
    
    # Create an organization with departments, projects and people
    memory.add({
        "@context": "https://schema.org/",
        "@id": "https://example.org/techcorp",
        "@type": "Organization",
        "name": "TechCorp",
        "description": "A technology company specializing in AI and semantic web",
        "department": [
            {"@id": "https://example.org/dept/engineering"},
            {"@id": "https://example.org/dept/research"}
        ],
        "foundingDate": "2010-01-15"
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "@id": "https://example.org/dept/engineering",
        "@type": "Organization",
        "name": "Engineering Department",
        "parentOrganization": {"@id": "https://example.org/techcorp"},
        "employee": [
            {"@id": "https://example.org/person/alice"},
            {"@id": "https://example.org/person/bob"}
        ]
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "@id": "https://example.org/dept/research",
        "@type": "Organization",
        "name": "Research Department",
        "parentOrganization": {"@id": "https://example.org/techcorp"},
        "employee": [
            {"@id": "https://example.org/person/charlie"},
            {"@id": "https://example.org/person/diana"}
        ],
        "project": {"@id": "https://example.org/project/semanticweb"}
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "@id": "https://example.org/person/alice",
        "@type": "Person",
        "name": "Alice Johnson",
        "jobTitle": "Senior Software Engineer",
        "worksFor": {"@id": "https://example.org/dept/engineering"},
        "knows": [
            {"@id": "https://example.org/person/bob"},
            {"@id": "https://example.org/person/charlie"}
        ]
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "@id": "https://example.org/person/bob",
        "@type": "Person",
        "name": "Bob Smith",
        "jobTitle": "DevOps Engineer",
        "worksFor": {"@id": "https://example.org/dept/engineering"}
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "@id": "https://example.org/person/charlie",
        "@type": "Person",
        "name": "Charlie Zhang",
        "jobTitle": "Research Scientist",
        "worksFor": {"@id": "https://example.org/dept/research"},
        "project": {"@id": "https://example.org/project/semanticweb"}
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "@id": "https://example.org/person/diana",
        "@type": "Person",
        "name": "Diana Rodriguez",
        "jobTitle": "Research Director",
        "worksFor": {"@id": "https://example.org/dept/research"},
        "project": {"@id": "https://example.org/project/semanticweb"},
        "knows": {"@id": "https://example.org/person/charlie"}
    })
    
    memory.add({
        "@context": "https://schema.org/",
        "@id": "https://example.org/project/semanticweb",
        "@type": "Project",
        "name": "Semantic Web Technologies",
        "description": "Research into advanced semantic web applications and knowledge graphs",
        "member": [
            {"@id": "https://example.org/person/charlie"},
            {"@id": "https://example.org/person/diana"}
        ]
    })
    
    print("✓ Created test knowledge graph with organizations, departments, people, and projects\n")
    
    # 2. Test semantic search with different query types
    print("2. Testing semantic search with different query types...")
    
    # Test direct ID lookup
    id_results = memory.semantic_search("https://example.org/techcorp", include_connected=False)
    print(f"Direct ID search results: {len(id_results)}")
    assert len(id_results) == 1, "Should find exactly one entity by ID"
    assert id_results[0]["@id"] == "https://example.org/techcorp", "Should find the correct entity"
    
    # Test type search
    type_results = memory.semantic_search("Person", include_connected=False)
    print(f"Type search results: {len(type_results)}")
    assert len(type_results) >= 4, "Should find all Person entities"
    
    # Test label/name search
    name_results = memory.semantic_search("Research", include_connected=False)
    print(f"Name search results: {len(name_results)}")
    assert len(name_results) >= 2, "Should find Research Department and related entities"
    
    # Test keyword search
    keyword_results = memory.semantic_search("semantic", include_connected=False)
    print(f"Keyword search results: {len(keyword_results)}")
    assert len(keyword_results) >= 2, "Should find entities with 'semantic' in their text"
    
    print("✓ Semantic search working for different query types\n")
    
    # 3. Test connected entity retrieval
    print("3. Testing connected entity retrieval...")
    
    # Test with depth 1
    connected_d1 = memory.semantic_search("https://example.org/techcorp", include_connected=True, max_depth=1)
    print(f"Connected search (depth 1) results: {len(connected_d1)}")
    assert len(connected_d1) > 1, "Should include connected entities at depth 1"
    
    # Test with depth 2
    connected_d2 = memory.semantic_search("https://example.org/techcorp", include_connected=True, max_depth=2)
    print(f"Connected search (depth 2) results: {len(connected_d2)}")
    assert len(connected_d2) > len(connected_d1), "Should include more entities at depth 2"
    
    print("✓ Connected entity retrieval working at different depths\n")
    
    # 4. Test LLM preparation
    print("4. Testing LLM preparation...")
    
    # Prepare a small subgraph for LLM
    llm_data = memory.search_for_llm("Charlie", include_connected=True, max_depth=1)
    
    # Check structure
    assert "entities" in llm_data, "LLM data should contain entities"
    assert "count" in llm_data, "LLM data should contain count"
    assert "relationships" in llm_data, "LLM data should contain relationships"
    
    print(f"LLM preparation generated {llm_data['count']} entities with {len(llm_data['relationships'])} relationships")
    
    # Check if relationships were extracted
    assert len(llm_data["relationships"]) > 0, "Should extract relationships between entities"
    
    # Check if vocabulary info was included
    if "vocabularies" in llm_data:
        print(f"Detected vocabularies: {', '.join(llm_data['vocabularies'].keys())}")
    
    print("✓ LLM preparation working correctly\n")
    
    # 5. Test relationship extraction
    print("5. Testing relationship extraction...")
    
    # Get a connected subgraph
    project_entities = memory.semantic_search("https://example.org/project/semanticweb", include_connected=True, max_depth=1)
    
    # Extract relationships
    relationships = memory._extract_relationships(project_entities)
    print(f"Extracted {len(relationships)} relationships from project subgraph")
    
    # Check relationship structure
    for rel in relationships[:3]:  # Show first 3 relationships
        print(f"  - {rel['source']} --{rel['property']}--> {rel['target']}")
    
    print("✓ Relationship extraction working correctly\n")
    
    print("=== All tests passed successfully ===")
    
    return memory, llm_data


In [ ]:
mem, llm_data = test_semantic_search_and_llm_prep()

=== Testing Semantic Search and LLM Preparation ===

1. Creating test knowledge graph...
✓ Created test knowledge graph with organizations, departments, people, and projects

2. Testing semantic search with different query types...
Searching for: https://example.org/techcorp
ID branch
Returning 1 results
Direct ID search results: 1
Searching for: Person
Type branch
Returning 0 results
Type search results: 0


AssertionError: Should find all Person entities

In [ ]:
@patch
def debug_mem(self:SemanticMemory):
    d={'by_id':self.indices.get("by_id",{}),
       'by_type':{k:len(v) for k,v in self.indices.get("by_type",{}).items()},
       'by_label':list(self.indices.get("by_label",{}).keys()),
       'by_kw':list(self.indices.get("by_keyword",{}).keys()),
       'ctx':list(self.context_registry.keys())}
    print("D-MEM:",f'by_id {len(d["by_id"])} keys {list(d["by_id"].keys())}',f'by_type {d["by_type"]}',f'by_label {d["by_label"]}',f'by_kw {d["by_kw"]}',f'ctx {d["ctx"]}')
    return d

In [ ]:
d = mem.debug_mem()

D-MEM: by_id 10 keys ['urn:test:entity:basic', 'urn:test:outbox:1', 'urn:test:complex:1', 'urn:test:subject:1', 'urn:uuid:704ef626-d88d-4088-8a83-5a47827ed4c2', 'urn:test:person:a', 'urn:test:person:b', 'urn:test:person:c', 'urn:test:person:d', 'urn:test:search:1'] by_type {'http://schema.org/Person': 7, 'https://www.w3.org/ns/activitystreams#OrderedCollection': 1, 'http://schema.org/Actor': 1, 'https://www.w3.org/2018/credentials#VerifiableCredential': 1, 'http://schema.org/Product': 1} by_label ['Basic Entity', 'Complex Entity', 'Credential Subject', 'Person A', 'Person B', 'Person C', 'Person D', 'Unique Search Test Product'] by_kw ['basic entity', 'basic', 'bas', 'basi', 'entity', 'ent', 'enti', 'entit', 'complex entity', 'complex', 'com', 'comp', 'compl', 'comple', 'credential subject', 'credential', 'cre', 'cred', 'crede', 'creden', 'credent', 'credenti', 'credentia', 'subject', 'sub', 'subj', 'subje', 'subjec', 'person a', 'person', 'per', 'pers', 'perso', 'person b', 'person c'

In [ ]:
def test_semantic_memory_new_structure():
    """Tests for semantic memory with proposed new structure functions"""
    import uuid, json
    from pathlib import Path
    
    # Setup
    cache_dir = Path("./test_cache")
    cache_dir.mkdir(exist_ok=True)
    memory = SemanticMemory(cache_dir=cache_dir)
    
    print("=== Testing New Semantic Memory Structure ===\n")
    
    # Test 1: Normalization functions
    print("Test 1: Entity normalization")
    test_entity = {
        "id": "urn:test:entity:1",
        "type": "Person",
        "name": "Test Person",
        "details": {"id": "urn:test:details:1", "type": "ContactInfo"}
    }
    
    try:
        # Test normalization
        normalized = memory._normalize_entity(test_entity)
        assert "@id" in normalized and "id" not in normalized, "ID not normalized"
        assert "@type" in normalized and "type" not in normalized, "Type not normalized"
        assert "@id" in normalized["details"] and "id" not in normalized["details"], "Nested ID not normalized"
        print("✓ Entity normalization working correctly")
    except Exception as e:
        print(f"✗ Entity normalization failed: {e}")
    
    # Test 2: Context registration
    print("\nTest 2: Context registration")
    test_context = {
        "@version": 1.1,
        "@vocab": "https://schema.org/",
        "nested": {
            "@id": "https://example.org/nested",
            "@context": {
                "@vocab": "https://example.org/vocab/"
            }
        }
    }
    
    try:
        # Register context
        ctx_id = memory._register_context(test_context)
        assert ctx_id is not None, "Context registration failed"
        assert ctx_id in memory.context_registry, "Context not found in registry"
        
        # Check scoped context extraction
        scoped_contexts = memory.context_registry[ctx_id]["scoped_contexts"]
        assert "nested" in scoped_contexts, "Scoped context not extracted"
        print("✓ Context registration with scoped contexts working")
    except Exception as e:
        print(f"✗ Context registration failed: {e}")
    
    # Test 3: Process entity
    print("\nTest 3: Entity processing")
    basic_entity = {
        "@context": "https://schema.org/",
        "@id": "urn:test:entity:basic",
        "@type": "Person",
        "name": "Basic Entity"
    }
    
    try:
        # Process single entity
        result = memory._process_entity(basic_entity)
        assert result is not None, "Entity processing failed"
        
        # Check if it was added to indices
        entity = memory.query_by_id("urn:test:entity:basic")
        assert entity is not None, "Entity not found in indices"
        print("✓ Entity processing working")
    except Exception as e:
        print(f"✗ Entity processing failed: {e}")
    
    # Test 4: Add method as single entry point
    print("\nTest 4: Add method as single entry point")
    complex_entity = {
        "@context": [
            "https://schema.org/",
            "https://www.w3.org/ns/activitystreams"
        ],
        "@id": "urn:test:complex:1",
        "@type": ["Person", "Actor"],
        "name": "Complex Entity",
        "outbox": {
            "@id": "urn:test:outbox:1",
            "@type": "OrderedCollection",
            "totalItems": 10
        }
    }
    
    try:
        # Add using the single entry point
        memory.add(complex_entity)
        
        # Check both entities were added
        entity1 = memory.query_by_id("urn:test:complex:1")
        entity2 = memory.query_by_id("urn:test:outbox:1")
        assert entity1 is not None, "Main entity not found"
        assert entity2 is not None, "Nested entity not found"
        print("✓ Add method working for complex entities")
    except Exception as e:
        print(f"✗ Add method failed: {e}")
    
    # Test 5: Graph partitioning and processing
    print("\nTest 5: Graph partitioning and processing")
    vc_data = {
        "@context": [
            "https://www.w3.org/ns/credentials/v2",
            "https://schema.org/"
        ],
        "@id": "urn:uuid:" + str(uuid.uuid4()),
        "@type": ["VerifiableCredential"],
        "issuer": "https://example.org/issuer",
        "credentialSubject": {
            "@id": "urn:test:subject:1",
            "@type": "Person",
            "name": "Credential Subject"
        }
    }
    
    try:
        # Test if it needs partitioning
        needs_partition = memory._needs_graph_partition(vc_data)
        assert needs_partition, "Failed to detect need for partitioning"
        
        # Create graph partition
        graph_data = memory._create_graph_partition(vc_data)
        assert "@graph" in graph_data, "Graph partitioning failed"
        assert len(graph_data["@graph"]) == 2, f"Expected 2 entities, got {len(graph_data['@graph'])}"
        
        # Process the graph
        results = memory._process_graph(graph_data)
        assert len(results) > 0, "Graph processing returned wrong number of entities"
        
        # Check that both entities were added
        credential = memory.query_by_id(vc_data["@id"])
        subject = memory.query_by_id("urn:test:subject:1")
        assert credential is not None, "Credential not found"
        assert subject is not None, "Subject not found"
        print("✓ Graph partitioning and processing working")
    except Exception as e:
        print(f"✗ Graph partitioning failed: {e}")
    
    # Test 6: Query and relationship traversal
    print("\nTest 6: Query and relationship traversal")
    try:
        # Create entities with relationships for testing
        memory.add({
            "@context": "https://schema.org/",
            "@id": "urn:test:person:a",
            "@type": "Person",
            "name": "Person A",
            "knows": [
                {"@id": "urn:test:person:b"},
                {"@id": "urn:test:person:c"}
            ]
        })
        
        memory.add({
            "@context": "https://schema.org/",
            "@id": "urn:test:person:b",
            "@type": "Person",
            "name": "Person B"
        })
        
        memory.add({
            "@context": "https://schema.org/",
            "@id": "urn:test:person:c",
            "@type": "Person",
            "name": "Person C",
            "knows": {"@id": "urn:test:person:d"}
        })
        
        memory.add({
            "@context": "https://schema.org/",
            "@id": "urn:test:person:d",
            "@type": "Person",
            "name": "Person D"
        })
        
        # Test connected entities retrieval
        connected = memory.get_connected_entities("urn:test:person:a")
        assert "urn:test:person:b" in connected, "Failed to find directly connected entity B"
        assert "urn:test:person:c" in connected, "Failed to find directly connected entity C"
        
        # Test with depth 2
        connected_d2 = memory.get_connected_entities("urn:test:person:a", max_depth=2)
        assert "urn:test:person:d" in connected_d2, "Failed to traverse to depth 2"
        print("✓ Relationship traversal working correctly")
    except Exception as e:
        print(f"✗ Relationship traversal failed: {e}")
    
    # Test 7: Search functionality
    print("\nTest 7: Search functionality")
    try:
        # Create entity for search testing
        search_test = {
            "@context": "https://schema.org/",
            "@id": "urn:test:search:1",
            "@type": "Product",
            "name": "Unique Search Test Product",
            "description": "A product specifically for testing search functionality"
        }
        
        memory.add(search_test)
        
        # Test search by name
        name_results = memory.search("Unique Search Test")
        assert len(name_results) > 0, "Name search failed"
        assert any(e.get("@id") == "urn:test:search:1" for e in name_results), "Product not found by name"
        
        # Test search by keyword
        keyword_results = memory.search("functionality")
        assert len(keyword_results) > 0, "Keyword search failed"
        assert any(e.get("@id") == "urn:test:search:1" for e in keyword_results), "Product not found by keyword"
        
        print("✓ Search functionality working correctly")
    except Exception as e:
        print(f"✗ Search functionality failed: {e}")
    
    # Final summary
    print("\n=== Final Stats ===")
    print(f"Total entities: {len(memory.indices['by_id'])}")
    print(f"Total contexts: {len(memory.context_registry)}")
    print(f"Entity types: {len(memory.indices['by_type'])}")
    
    return memory


In [ ]:
mem = test_semantic_memory_new_structure()


=== Testing New Semantic Memory Structure ===

Test 1: Entity normalization
✓ Entity normalization working correctly

Test 2: Context registration
✓ Context registration with scoped contexts working

Test 3: Entity processing
✓ Entity processing working

Test 4: Add method as single entry point
✓ Add method working for complex entities

Test 5: Graph partitioning and processing
✓ Graph partitioning and processing working

Test 6: Query and relationship traversal
✓ Relationship traversal working correctly

Test 7: Search functionality
✓ Search functionality working correctly

=== Final Stats ===
Total entities: 10
Total contexts: 5
Entity types: 5


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()